# STA 554 Final Project
Jillian Greene

This project will assess our ability to use spark for data streaming, machine learning, and more.

In [5]:
# Import packages
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, Binarizer, StringIndexer, OneHotEncoder, VectorAssembler, PCA
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

In [2]:
# Read in power_ml_data with pandas
power_ml_pd = pd.read_csv("power_ml_data.csv")

power_ml_pd.head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


In [3]:
# Initialize Spark
spark = SparkSession.builder.getOrCreate()

# Convert to spark dataframe
power_ml_sp = spark.createDataFrame(power_ml_pd)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/22 09:55:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


We want to fit an elastic net model using CV (no training/test split, just using CV on the data we’ve read in). Imagine we know that the Power_Zone_3 reading
is going to go offline in the future and we need to be able to predict that value appropriately.

### Transformations

In [8]:
# Recast Hour
cast_hour = SQLTransformer(statement="""
    SELECT *, CAST(Hour AS DOUBLE) as Hour_double FROM __THIS__
    """)

# Binarize Hour for night and day
binarizer = Binarizer(inputCol="Hour_double", outputCol="Hour_binary", threshold=6.5)

# One-hot encode Month (a function for this! I've always done it messily myself)
month_indexer = StringIndexer(inputCol="Month", outputCol="Month_index", handleInvalid="keep")

month_encoder = OneHotEncoder(inputCol="Month_index", outputCol="Month_ohe")

# PCA on climate and diffuse flows cols
pca_features = ["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows"]

pca_assembler = VectorAssembler(inputCols=pca_features, outputCol="pca_input")

pca = PCA(k=2, inputCol="pca_input", outputCol="pca_features")

# Recode Power_Zone_3 as the label
rename_label = SQLTransformer(statement="""
    SELECT *, Power_Zone_3 as label FROM __THIS__
    """)

# Assemble features
final_features = VectorAssembler(
    inputCols=["pca_features", "Hour_binary", "Power_Zone_1", "Power_Zone_2", "Month_ohe"],
    outputCol="features")

### Set up model pipeline

In [9]:
# Initialize model
lr = LinearRegression(featuresCol="features", labelCol="label")

# Set up parameter grid for tuning with req. combinations
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()

# Evaluator for RMSE
evaluator = RegressionEvaluator(labelCol="label",
    predictionCol="prediction",
    metricName="rmse")

# Pipeline
pipeline = Pipeline(stages=[cast_hour, binarizer, month_indexer, month_encoder, pca_assembler,
                            pca, rename_label, final_features])

### Cross Validation

In [10]:
cv = CrossValidator(estimator=Pipeline(stages=pipeline.getStages() + [lr]),
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=5)

### Fit Model

This will take a long time, since the parameter grid is so large

In [11]:
cv_model = cv.fit(power_ml_sp)

26/04/22 10:17:20 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/22 10:17:20 WARN Instrumentation: [6b777641] regParam is zero, which might cause numerical instability and overfitting.
26/04/22 10:17:21 WARN Instrumentation: [6b777641] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/22 10:17:23 WARN Instrumentation: [45bde3b0] regParam is zero, which might cause numerical instability and overfitting.
26/04/22 10:17:25 WARN Instrumentation: [21376181] regParam is zero, which might cause numerical instability and overfitting.
26/04/22 10:17:25 WARN Instrumentation: [21376181] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/22 10:17:26 WARN Instrumentation: [4bb49cc4] regParam is zero, which might cause numerical instability and overfitting.
26/04/22 10:17:27 W

In [12]:
# Extract best model and stats
best_model = cv_model.bestModel.stages[-1]

print("Best regParam:", best_model._java_obj.getRegParam())
print("Best elasticNetParam:", best_model._java_obj.getElasticNetParam())
print("CV RMSE:", min(cv_model.avgMetrics))

Best regParam: 0.9
Best elasticNetParam: 0.5
CV RMSE: 2147.8469960549196


### Apply model as transformer for training RMSE

In [13]:
predictions = cv_model.transform(power_ml_sp)

train_rmse = evaluator.evaluate(predictions)
print("Training RMSE:", train_rmse)

Training RMSE: 2147.0983043611513


In [14]:
# Create and print residuals column
results_df = predictions.withColumn("residual",
    F.col("label") - F.col("prediction")).select("label", "prediction", "residual")

results_df.show(20)

+-----------+------------------+------------------+
|      label|        prediction|          residual|
+-----------+------------------+------------------+
|20240.96386|20874.584433099557|-633.6205730995571|
|20131.08434|18655.869200591842|1475.2151394081593|
|19668.43373|18200.572066745324|1467.8616632546764|
|18899.27711|17586.736619947944|1312.5404900520552|
|18442.40964|16993.623736135083|1448.7859038649185|
|18130.12048|16514.208856797253|1615.9116232027482|
|17945.06024|16089.949343959703|1855.1108960402962|
|17459.27711|15719.549474610747| 1739.727635389252|
|17025.54217| 15268.08747450489| 1757.454695495111|
|16794.21687|14935.525953623954|1858.6909163760465|
|16638.07229|14649.724403084943|1988.3478869150567|
|16395.18072| 14412.33985890693| 1982.840861093071|
|16117.59036|14080.415519429222|2037.1748405707785|
| 15822.6506|  13622.6042694511| 2200.046330548901|
|15672.28916|13448.159148753448|2224.1300112465524|
|15597.10843|13300.160129229356| 2296.948300770644|
|15510.36145

Based on the overall RMSE and the residuals printed above, the model seems to underpredict the `Power_Level_3` more o